In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import (
    shapiro,
    pearsonr,
    spearmanr,
    mannwhitneyu,
    ttest_ind,
    ttest_rel,
    wilcoxon,
    chi2_contingency
)
MAIN_FOLDER = "../"

In [ ]:
# ============================================================
# 1. Load dataset
# ============================================================
# 1. Load dataset
FILE_NAME = "patient_list_with_ids"

# Read the excel file
df = pd.read_excel("../../"+FILE_NAME+".xlsx")

# Show the head
print(df.tail())

In [ ]:
# ============================================================
# 2. Create output folders
# ============================================================

output_folder = MAIN_FOLDER+"2-patient_and_tooth_level_statistics"
figures_folder = os.path.join(output_folder, "figures")
tables_folder = os.path.join(output_folder, "tables")

os.makedirs(output_folder, exist_ok=True)
os.makedirs(figures_folder, exist_ok=True)
os.makedirs(tables_folder, exist_ok=True)

In [ ]:
# ============================================================
# 3. Basic cleaning
# ============================================================

# Expected columns:
# name, pell_gregory, pederson, winter_angulation, age, gender_tr, gender_eng

# Remove leading/trailing spaces from text columns
text_columns = ["name", "pell_gregory", "winter_angulation", "gender_tr", "gender_eng"]

for col in text_columns:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

# Standardize Pell-Gregory
df["pell_gregory"] = df["pell_gregory"].str.upper()

# Standardize Winter angulation
df["winter_angulation"] = (
    df["winter_angulation"]
    .str.lower()
    .str.replace("_", " ")
    .str.title()
)

# Standardize gender values
df["gender_eng"] = (
    df["gender_eng"]
    .astype(str)
    .str.strip()
    .str.lower()
)

# Convert numerical columns
df["age"] = pd.to_numeric(df["age"], errors="coerce")
df["pederson"] = pd.to_numeric(df["pederson"], errors="coerce")

In [ ]:
# ============================================================
# 4. Extract patient ID and side information
# ============================================================

# Extract patient ID from values such as:
# 1-SOL ALT, 1-SAĞ ALT, 2-SOL ALT, 2-SAĞ ALT
#df["patient_id"] = (
#    df["name"]
#    .astype(str)
#    .str.extract(r"^\s*(\d+)")
#)

# Extract side from name
def extract_side(name):
    """
    Extract side information from the name column.
    """
    name = str(name).upper().strip()

    if "SOL" in name:
        return "left"
    elif "SAĞ" in name or "SAG" in name:
        return "right"
    else:
        return "unknown"

df["side"] = df["name"].apply(extract_side)

print("\nSide counts:")
print(df["side"].value_counts(dropna=False))

In [ ]:
# ============================================================
# 5. Create patient-level dataset
# ============================================================

# Patient-level dataset is used for age and gender statistics.
# Each patient should appear only once.
patient_df = (
    df.sort_values(["id", "side"])
    .drop_duplicates(subset=["id"])
    .copy()
)

print("\nNumber of tooth-level records:", len(df))
print("Number of unique patients:", patient_df["id"].nunique())

patient_df.to_excel(
    os.path.join(output_folder, "patient_level_dataset.xlsx"),
    index=False
)

df.to_excel(
    os.path.join(output_folder, "tooth_level_dataset_cleaned.xlsx"),
    index=False
)

In [ ]:
# ============================================================
# 6. Helper functions
# ============================================================

def save_table(table, filename):
    """
    Save a table as CSV and Excel.
    """
    csv_path = os.path.join(tables_folder, filename + ".csv")
    excel_path = os.path.join(tables_folder, filename + ".xlsx")

    table.to_csv(csv_path)
    table.to_excel(excel_path)

    print(f"Saved table: {csv_path}")
    print(f"Saved table: {excel_path}")


def frequency_table(data, column, order=None):
    """
    Create frequency and percentage table.
    """
    if order is not None:
        counts = data[column].value_counts(dropna=False).reindex(order, fill_value=0)
    else:
        counts = data[column].value_counts(dropna=False)

    percentages = counts / counts.sum() * 100

    table = pd.DataFrame({
        "Count": counts,
        "Percentage (%)": percentages.round(2)
    })

    return table


def my_autopct(pct):
    """
    Hide very small percentages in pie chart.
    """
    return ("%1.1f%%" % pct) if pct > 1.0 else ""


In [ ]:
def plot_bar_chart(counts, title, xlabel, ylabel, filename):
    """
    Create and save colored bar chart.
    """
    counts_nonzero = counts[counts > 0]

    # ---------------------------
    # Global style settings for academic appearance
    # ---------------------------
    plt.rcParams.update({
        "font.family": "serif",
        "font.serif": ["Times New Roman", "DejaVu Serif", "serif"],
        "font.size": 12,
        "axes.titlesize": 14,
        "axes.labelsize": 12,
        "xtick.labelsize": 11,
        "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "figure.dpi": 300,
        "savefig.dpi": 300
    })

    # ---------------------------
    # Figure
    # ---------------------------
    fig, ax = plt.subplots(figsize=(9, 6))

    #colors = plt.cm.Set3(np.linspace(0, 1, len(counts_nonzero)))
    colors = plt.cm.Blues(np.linspace(0.45, 0.85, len(counts_nonzero)))

    bars = ax.bar(
        counts_nonzero.index.astype(str),
        counts_nonzero.values,
        color=colors,
        edgecolor="black",
        linewidth=0.8
    )

    ax.set_title(title)
    ax.set_xlabel(xlabel, labelpad=10)
    ax.set_ylabel(ylabel, labelpad=10)
    #plt.xticks(rotation=0, ha="center")

    # ---------------------------
    # Axis formatting
    # ---------------------------
    ax.tick_params(axis="x", rotation=0)
    for tick in ax.get_xticklabels():
        tick.set_horizontalalignment("center")

    # Remove top and right spines for a cleaner academic look
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # Add light horizontal gridlines
    ax.yaxis.grid(True, linestyle="--", linewidth=0.6, alpha=0.5)
    ax.set_axisbelow(True)

    # ---------------------------
    # Add values on top of bars
    # ---------------------------
    y_max = counts_nonzero.values.max()
    ax.set_ylim(0, y_max * 1.15)

    for bar in bars:
        height = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            height + (y_max * 0.02),
            f"{int(height)}",
            ha="center",
            va="bottom",
            fontsize=11
        )

    
    # ---------------------------
    # Layout and save
    # ---------------------------
    plt.tight_layout()

    bar_path = os.path.join(figures_folder, filename)
    plt.savefig(
        bar_path,
        dpi=300,
        bbox_inches="tight",
        facecolor="white"
    )

    plt.show()
    plt.close()

    print(f"Saved: {bar_path}")
    #path = os.path.join(figures_folder, filename)
    #plt.savefig(path, dpi=300, bbox_inches="tight")
    #plt.show()
    #plt.close()

    #print(f"Saved figure: {path}")

In [ ]:
def plot_pie_chart(counts, title, legend_title, filename):
    """
    Create and save donut pie chart.
    """
    counts_nonzero = counts[counts > 0]


    # ---------------------------
    # Global style settings
    # ---------------------------
    plt.rcParams.update({
        "font.family": "serif",
        "font.serif": ["Times New Roman", "DejaVu Serif", "serif"],
        "font.size": 12,
        "axes.titlesize": 14,
        "axes.labelsize": 12,
        "legend.fontsize": 12,
        "legend.title_fontsize": 12,
        "figure.dpi": 300,
        "savefig.dpi": 300
    })

    plt.figure(figsize=(9, 9))

    #colors = plt.cm.Pastel1(np.linspace(0, 1, len(counts_nonzero)))
    colors = plt.cm.Blues(np.linspace(0.45, 0.85, len(counts_nonzero)))

    wedges, texts, autotexts = plt.pie(
        counts_nonzero,
        autopct=my_autopct,
        startangle=90,
        #pctdistance=0.85,
        colors=colors,
        wedgeprops=dict(
            edgecolor="white",
            linewidth=1.2
        )
        #wedgeprops=dict(width=0.5, edgecolor="w")
    )

    plt.title(title)

    plt.legend(
        wedges,
        counts_nonzero.index.astype(str),
        title=legend_title,
        loc="center left",
        bbox_to_anchor=(1, 0, 0.5, 1)
    )

    plt.tight_layout()

    path = os.path.join(figures_folder, filename)
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

    print(f"Saved figure: {path}")

In [ ]:
# ============================================================
# 7. Gender ratio - patient-level
# ============================================================

gender_order = ["female", "male"]

gender_table = frequency_table(
    patient_df,
    "gender_eng",
    order=gender_order
)

print("\nGender Ratio - Patient Level")
print(gender_table)

save_table(gender_table, "gender_ratio_patient_level")



In [ ]:
plot_bar_chart(
    gender_table["Count"],
    "Distribution of Gender",
    "Gender",
    "Number of Patients",
    "gender_bar_chart_patient_level.png"
)

In [ ]:
plot_pie_chart(
    gender_table["Count"],
    "Percentage Distribution of Gender",
    "Gender",
    "gender_donut_chart_patient_level.png"
)

In [ ]:
# ============================================================
# 8. Age statistics - patient-level
# ============================================================

age_stats = patient_df["age"].describe().to_frame(name="Age")

print("\nAge Descriptive Statistics - Patient Level")
print(age_stats)

save_table(age_stats, "age_descriptive_statistics_patient_level")

# Age groups
age_bins = [0, 20, 30, 40, 50, 100]
age_labels = ["≤20", "21–30", "31–40", "41–50", "51+"]

patient_df["age_group"] = pd.cut(
    patient_df["age"],
    bins=age_bins,
    labels=age_labels,
    right=True
)

age_group_table = frequency_table(
    patient_df,
    "age_group",
    order=age_labels
)

print("\nAge Group Distribution - Patient Level")
print(age_group_table)

save_table(age_group_table, "age_group_distribution_patient_level")

In [ ]:
plot_bar_chart(
    age_group_table["Count"],
    "Distribution of Age Groups",
    "Age Group",
    "Number of Patients",
    "age_group_bar_chart_patient_level.png"
)

In [ ]:
plot_pie_chart(
    age_group_table["Count"],
    "Percentage Distribution of Age Groups",
    "Age Group",
    "age_group_donut_chart_patient_level.png"
)

In [ ]:
# Age histogram
plt.figure(figsize=(9, 6))
plt.hist(patient_df["age"].dropna(), bins=10, edgecolor="black")
plt.title("Age Distribution")
plt.xlabel("Age")
plt.ylabel("Number of Patients")
plt.tight_layout()

path = os.path.join(figures_folder, "age_histogram_patient_level.png")
plt.savefig(path, dpi=300, bbox_inches="tight")
plt.show()
plt.close()

In [ ]:
# ============================================================
# 9. Tooth-level descriptive statistics by side
# ============================================================

side_pederson_stats = df.groupby("side")["pederson"].agg(
    ["count", "mean", "std", "median", "min", "max"]
)

print("\nPederson Descriptive Statistics by Side")
print(side_pederson_stats)

save_table(side_pederson_stats, "pederson_statistics_by_side")

In [ ]:
# ============================================================
# 10. Left and right separate datasets
# ============================================================

left_df = df[df["side"] == "left"].copy()
right_df = df[df["side"] == "right"].copy()

left_df.to_excel(
    os.path.join(output_folder, "left_tooth_dataset.xlsx"),
    index=False
)

right_df.to_excel(
    os.path.join(output_folder, "right_tooth_dataset.xlsx"),
    index=False
)

print("\nLeft tooth records:", len(left_df))
print("Right tooth records:", len(right_df))


In [ ]:
# ============================================================
# 11. Separate statistics for left and right teeth
# ============================================================

pell_order = ["1A", "1B", "1C", "2A", "2B", "2C", "3A", "3B", "3C"]

for side_name, side_data in [("left", left_df), ("right", right_df)]:

    print(f"\nStatistics for {side_name.capitalize()} Tooth")

    # Pell-Gregory distribution
    pell_table = frequency_table(
        side_data,
        "pell_gregory",
        order=pell_order
    )

    print(f"\nPell-Gregory Distribution - {side_name.capitalize()}")
    print(pell_table)

    save_table(
        pell_table,
        f"pell_gregory_distribution_{side_name}"
    )

    plot_bar_chart(
        pell_table["Count"],
        f"Distribution of Pell-Gregory Classes - {side_name.capitalize()} Tooth",
        "Pell-Gregory Class",
        "Number of Cases",
        f"pell_gregory_bar_chart_{side_name}.png"
    )

    plot_pie_chart(
        pell_table["Count"],
        f"Percentage Distribution of Pell-Gregory Classes - {side_name.capitalize()} Tooth",
        "Pell-Gregory Class",
        f"pell_gregory_donut_chart_{side_name}.png"
    )

    # Winter angulation distribution
    winter_table = frequency_table(
        side_data,
        "winter_angulation"
    )

    print(f"\nWinter Angulation Distribution - {side_name.capitalize()}")
    print(winter_table)

    save_table(
        winter_table,
        f"winter_angulation_distribution_{side_name}"
    )

    plot_bar_chart(
        winter_table["Count"],
        f"Distribution of Winter Angulation Types - {side_name.capitalize()} Tooth",
        "Winter Angulation Type",
        "Number of Cases",
        f"winter_angulation_bar_chart_{side_name}.png"
    )

    plot_pie_chart(
        winter_table["Count"],
        f"Percentage Distribution of Winter Angulation Types - {side_name.capitalize()} Tooth",
        "Winter Angulation Type",
        f"winter_angulation_donut_chart_{side_name}.png"
    )

    # Pederson score distribution
    pederson_table = frequency_table(
        side_data.dropna(subset=["pederson"]),
        "pederson"
    )

    print(f"\nPederson Score Distribution - {side_name.capitalize()}")
    print(pederson_table)

    save_table(
        pederson_table,
        f"pederson_distribution_{side_name}"
    )

    plot_bar_chart(
        pederson_table["Count"],
        f"Distribution of Pederson Scores - {side_name.capitalize()} Tooth",
        "Pederson Score",
        "Number of Cases",
        f"pederson_bar_chart_{side_name}.png"
    )

    plot_pie_chart(
        pederson_table["Count"],
        f"Percentage Distribution of Pederson Scores - {side_name.capitalize()} Tooth",
        "Pederson Score",
        f"pederson_donut_chart_{side_name}.png"
    )



In [ ]:
# ============================================================
# 12. Pederson and age relationship
# ============================================================

# Because each patient may have left and right records, we calculate:
# 1) tooth-level relationship
# 2) patient-level average Pederson relationship

# Tooth-level relationship
ped_age_tooth = df[["age", "pederson"]].dropna()

if len(ped_age_tooth) > 2:
    pearson_stat, pearson_p = pearsonr(
        ped_age_tooth["age"],
        ped_age_tooth["pederson"]
    )

    spearman_stat, spearman_p = spearmanr(
        ped_age_tooth["age"],
        ped_age_tooth["pederson"]
    )

    ped_age_tooth_result = pd.DataFrame({
        "Analysis Level": ["Tooth-level", "Tooth-level"],
        "Test": ["Pearson correlation", "Spearman correlation"],
        "Statistic": [pearson_stat, spearman_stat],
        "p-value": [pearson_p, spearman_p]
    })

    print("\nPederson and Age Relationship - Tooth Level")
    print(ped_age_tooth_result)

    save_table(
        ped_age_tooth_result,
        "pederson_age_relationship_tooth_level"
    )

# Patient-level average Pederson
patient_pederson_mean = (
    df.groupby("id")
    .agg(
        age=("age", "first"),
        gender_eng=("gender_eng", "first"),
        mean_pederson=("pederson", "mean")
    )
    .reset_index()
)

ped_age_patient = patient_pederson_mean[["age", "mean_pederson"]].dropna()

if len(ped_age_patient) > 2:
    pearson_stat, pearson_p = pearsonr(
        ped_age_patient["age"],
        ped_age_patient["mean_pederson"]
    )

    spearman_stat, spearman_p = spearmanr(
        ped_age_patient["age"],
        ped_age_patient["mean_pederson"]
    )

    ped_age_patient_result = pd.DataFrame({
        "Analysis Level": ["Patient-level", "Patient-level"],
        "Test": ["Pearson correlation", "Spearman correlation"],
        "Statistic": [pearson_stat, spearman_stat],
        "p-value": [pearson_p, spearman_p]
    })

    print("\nPederson and Age Relationship - Patient Level")
    print(ped_age_patient_result)

    save_table(
        ped_age_patient_result,
        "pederson_age_relationship_patient_level"
    )

# Scatter plot: age vs Pederson, tooth-level
plt.figure(figsize=(8, 6))
plt.scatter(ped_age_tooth["age"], ped_age_tooth["pederson"], alpha=0.7)
plt.title("Relationship Between Age and Pederson Score")
plt.xlabel("Age")
plt.ylabel("Pederson Score")
plt.tight_layout()

path = os.path.join(figures_folder, "scatter_age_vs_pederson_tooth_level.png")
plt.savefig(path, dpi=300, bbox_inches="tight")
plt.show()
plt.close()



In [ ]:

# ============================================================
# 13. Pederson and gender relationship
# ============================================================

# Tooth-level comparison
ped_gender_tooth = df[["gender_eng", "pederson"]].dropna()

female_scores_tooth = ped_gender_tooth.loc[
    ped_gender_tooth["gender_eng"] == "female",
    "pederson"
]

male_scores_tooth = ped_gender_tooth.loc[
    ped_gender_tooth["gender_eng"] == "male",
    "pederson"
]

ped_gender_results = []

if len(female_scores_tooth) > 1 and len(male_scores_tooth) > 1:

    # Independent t-test
    t_stat, t_p = ttest_ind(
        female_scores_tooth,
        male_scores_tooth,
        equal_var=False,
        nan_policy="omit"
    )

    # Mann-Whitney U test
    u_stat, u_p = mannwhitneyu(
        female_scores_tooth,
        male_scores_tooth,
        alternative="two-sided"
    )

    ped_gender_results.append({
        "Analysis Level": "Tooth-level",
        "Test": "Welch independent t-test",
        "Statistic": t_stat,
        "p-value": t_p
    })

    ped_gender_results.append({
        "Analysis Level": "Tooth-level",
        "Test": "Mann-Whitney U test",
        "Statistic": u_stat,
        "p-value": u_p
    })

# Patient-level comparison using mean Pederson per patient
ped_gender_patient = patient_pederson_mean[["gender_eng", "mean_pederson"]].dropna()

female_scores_patient = ped_gender_patient.loc[
    ped_gender_patient["gender_eng"] == "female",
    "mean_pederson"
]

male_scores_patient = ped_gender_patient.loc[
    ped_gender_patient["gender_eng"] == "male",
    "mean_pederson"
]

if len(female_scores_patient) > 1 and len(male_scores_patient) > 1:

    t_stat, t_p = ttest_ind(
        female_scores_patient,
        male_scores_patient,
        equal_var=False,
        nan_policy="omit"
    )

    u_stat, u_p = mannwhitneyu(
        female_scores_patient,
        male_scores_patient,
        alternative="two-sided"
    )

    ped_gender_results.append({
        "Analysis Level": "Patient-level",
        "Test": "Welch independent t-test",
        "Statistic": t_stat,
        "p-value": t_p
    })

    ped_gender_results.append({
        "Analysis Level": "Patient-level",
        "Test": "Mann-Whitney U test",
        "Statistic": u_stat,
        "p-value": u_p
    })

ped_gender_results_df = pd.DataFrame(ped_gender_results)

print("\nPederson and Gender Relationship")
print(ped_gender_results_df)

save_table(
    ped_gender_results_df,
    "pederson_gender_relationship"
)

# Boxplot: Pederson by gender, tooth-level
plt.figure(figsize=(7, 6))
df.boxplot(column="pederson", by="gender_eng", grid=False)
plt.title("Pederson Score by Gender")
plt.suptitle("")
plt.xlabel("Gender")
plt.ylabel("Pederson Score")
plt.tight_layout()

path = os.path.join(figures_folder, "boxplot_pederson_by_gender_tooth_level.png")
plt.savefig(path, dpi=300, bbox_inches="tight")
plt.show()
plt.close()

In [ ]:
# ============================================================
# 14. Left Pederson vs right Pederson comparison
# ============================================================

# Create paired left-right Pederson dataset
paired_pederson = df.pivot_table(
    index="id",
    columns="side",
    values="pederson",
    aggfunc="first"
).reset_index()

# Keep only patients with both left and right Pederson scores
paired_pederson = paired_pederson.dropna(subset=["left", "right"]).copy()

paired_pederson["difference_left_minus_right"] = (
    paired_pederson["left"] - paired_pederson["right"]
)

paired_pederson.to_excel(
    os.path.join(tables_folder, "paired_left_right_pederson.xlsx"),
    index=False
)

print("\nPaired Left-Right Pederson Dataset")
print(paired_pederson.head())

paired_stats = paired_pederson[["left", "right", "difference_left_minus_right"]].describe()

print("\nPaired Left-Right Pederson Descriptive Statistics")
print(paired_stats)

save_table(
    paired_stats,
    "paired_left_right_pederson_descriptive_statistics"
)

left_paired = paired_pederson["left"]
right_paired = paired_pederson["right"]
diff_paired = paired_pederson["difference_left_minus_right"]

left_right_results = []

if len(paired_pederson) > 2:

    # Test normality of differences
    shapiro_stat, shapiro_p = shapiro(diff_paired)

    left_right_results.append({
        "Test": "Shapiro-Wilk normality test for left-right differences",
        "Statistic": shapiro_stat,
        "p-value": shapiro_p
    })

    # Paired t-test
    t_stat, t_p = ttest_rel(left_paired, right_paired)

    left_right_results.append({
        "Test": "Paired t-test",
        "Statistic": t_stat,
        "p-value": t_p
    })

    # Wilcoxon signed-rank test
    # Wilcoxon is useful when the difference is not normally distributed.
    try:
        w_stat, w_p = wilcoxon(left_paired, right_paired)

        left_right_results.append({
            "Test": "Wilcoxon signed-rank test",
            "Statistic": w_stat,
            "p-value": w_p
        })

    except ValueError as e:
        left_right_results.append({
            "Test": "Wilcoxon signed-rank test",
            "Statistic": np.nan,
            "p-value": np.nan
        })

left_right_results_df = pd.DataFrame(left_right_results)

print("\nLeft vs Right Pederson Comparison")
print(left_right_results_df)

save_table(
    left_right_results_df,
    "left_vs_right_pederson_comparison"
)



In [ ]:

# Boxplot for paired left-right Pederson scores
plt.figure(figsize=(7, 6))
plt.boxplot(
    [left_paired, right_paired],
    tick_labels=["Left", "Right"]
)
plt.title("Left vs Right Pederson Scores")
plt.xlabel("Side")
plt.ylabel("Pederson Score")
plt.tight_layout()

path = os.path.join(figures_folder, "boxplot_left_vs_right_pederson.png")
plt.savefig(path, dpi=300, bbox_inches="tight")
plt.show()
plt.close()

# Scatter plot for left vs right Pederson scores
plt.figure(figsize=(7, 7))
plt.scatter(left_paired, right_paired, alpha=0.7)

min_value = min(left_paired.min(), right_paired.min())
max_value = max(left_paired.max(), right_paired.max())

plt.plot([min_value, max_value], [min_value, max_value], linestyle="--")

plt.title("Left and Right Pederson Score Agreement")
plt.xlabel("Left Pederson Score")
plt.ylabel("Right Pederson Score")
plt.tight_layout()

path = os.path.join(figures_folder, "scatter_left_vs_right_pederson.png")
plt.savefig(path, dpi=300, bbox_inches="tight")
plt.show()
plt.close()

In [ ]:
# ============================================================
# 15. Left-right categorical comparisons
# ============================================================

# Compare Pell-Gregory distribution between left and right sides
side_pell_table = pd.crosstab(df["side"], df["pell_gregory"])

print("\nSide vs Pell-Gregory Cross-tabulation")
print(side_pell_table)

save_table(
    side_pell_table,
    "side_vs_pell_gregory_crosstab"
)

if side_pell_table.shape[0] > 1 and side_pell_table.shape[1] > 1:
    chi2, p, dof, expected = chi2_contingency(side_pell_table)

    side_pell_chi = pd.DataFrame({
        "Test": ["Chi-square test"],
        "Chi-square": [chi2],
        "p-value": [p],
        "Degrees of freedom": [dof]
    })

    print("\nSide vs Pell-Gregory Chi-square Test")
    print(side_pell_chi)

    save_table(
        side_pell_chi,
        "side_vs_pell_gregory_chi_square"
    )

# Compare Winter angulation distribution between left and right sides
side_winter_table = pd.crosstab(df["side"], df["winter_angulation"])

print("\nSide vs Winter Angulation Cross-tabulation")
print(side_winter_table)

save_table(
    side_winter_table,
    "side_vs_winter_angulation_crosstab"
)

if side_winter_table.shape[0] > 1 and side_winter_table.shape[1] > 1:
    chi2, p, dof, expected = chi2_contingency(side_winter_table)

    side_winter_chi = pd.DataFrame({
        "Test": ["Chi-square test"],
        "Chi-square": [chi2],
        "p-value": [p],
        "Degrees of freedom": [dof]
    })

    print("\nSide vs Winter Angulation Chi-square Test")
    print(side_winter_chi)

    save_table(
        side_winter_chi,
        "side_vs_winter_angulation_chi_square"
    )


In [ ]:
# ============================================================
# 16. Final summary
# ============================================================

print("\nAnalysis completed.")
print(f"All tables saved to: {tables_folder}")
print(f"All figures saved to: {figures_folder}")
print(f"Patient-level dataset saved to: {os.path.join(output_folder, 'patient_level_dataset.xlsx')}")
print(f"Tooth-level dataset saved to: {os.path.join(output_folder, 'tooth_level_dataset_cleaned.xlsx')}")